# Ideal VQE Simulation

In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np

In [5]:
!pip install --upgrade qiskit

In [4]:
from qiskit.version import VERSION
print(VERSION)

2.1.1


In [ ]:
import qiskit
print(qiskit.__version__)

2.1.1


In [4]:
!pip install -U qiskit


In [6]:
import qiskit
print(qiskit.__version__)


2.1.1


In [8]:
!pip install qiskit-algorithms

  Using cached qiskit_algorithms-0.3.1-py3-none-any.whl.metadata (4.2 kB)
Using cached qiskit_algorithms-0.3.1-py3-none-any.whl (310 kB)


In [ ]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.minimum_eigensolvers import VQE

from qiskit.primitives import Estimator


hamiltonian = SparsePauliOp.from_list([("II", -1.052), ("IZ", 0.398), ("ZI", -0.398), ("ZZ", -0.011), ("XX", 0.181)])


ansatz = TwoLocal(num_qubits=2, rotation_blocks='ry', entanglement_blocks='cx', reps=2)


optimizer = SPSA(maxiter=100)


estimator = Estimator()


vqe = VQE(ansatz=ansatz, optimizer=optimizer, estimator=estimator)


result = vqe.compute_minimum_eigenvalue(operator=hamiltonian)


print("Computed ground state energy:", result.eigenvalue.real)


I don't know why but at last my libraries are not imported.

# Part2

In [ ]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer import AerSimulator


hamiltonian = SparsePauliOp.from_list([("II", -1.052), ("IZ", 0.398), ("ZI", -0.398), ("ZZ", -0.011), ("XX", 0.181)])


ansatz = TwoLocal(num_qubits=2, rotation_blocks='ry', entanglement_blocks='cx', reps=2)
optimizer = SPSA(maxiter=100)


bit_flip_error = pauli_error([('X', 0.05), ('I', 0.95)])
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(bit_flip_error, ['cx'])


backend = AerSimulator(noise_model=noise_model)
noisy_estimator = AerEstimator(backend=backend)


vqe_noisy = VQE(ansatz=ansatz, optimizer=optimizer, estimator=noisy_estimator)
result_noisy = vqe_noisy.compute_minimum_eigenvalue(operator=hamiltonian)


print("Noisy ground state energy:", result_noisy.eigenvalue.real)


# Part3

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, Aer, transpile, assemble


logical = QuantumRegister(1, 'logical')         
data = QuantumRegister(3, 'data')               
anc = QuantumRegister(2, 'ancilla')             
creg = ClassicalRegister(2, 'syndrome')         
qc = QuantumCircuit(logical, data, anc, creg)


qc.x(logical[0])


encode_bit_flip(qc, logical, data)


qc.x(data[1])  


add_syndrome_measurement(qc, data, anc, creg)


add_correction_circuit(qc, data, creg)


qc.measure_all()


backend = Aer.get_backend('qasm_simulator')
compiled = transpile(qc, backend)
qobj = assemble(compiled)
result = backend.run(qobj).result()
counts = result.get_counts()

print("Final Measurement Counts:", counts)


# Part4

In [ ]:
def encode(qc, logical, physical):
    qc.cx(logical[0], physical[0])
    qc.cx(logical[0], physical[1])
    qc.cx(logical[0], physical[2])

def decode(qc, logical, physical):
    qc.cx(logical[0], physical[1])
    qc.cx(logical[0], physical[2])
    qc.cx(logical[0], physical[0])


In [ ]:
from qiskit.circuit import Parameter

def protected_ansatz(params, physical):
    qc = QuantumCircuit(physical)
    reps = len(params) // 3  # assuming 3 params per rep (for 3 Ry rotations)
    idx = 0
    for r in range(reps):
        # Apply Ry on all physical qubits with same param
        for i in range(3):
            qc.ry(params[idx], physical[i])
            idx += 1
        # Apply entangling CNOTs
        qc.cx(physical[0], physical[1])
        qc.cx(physical[1], physical[2])
    return qc


In [ ]:
def measure_and_correct(qc, data, ancilla, creg):
    qc.cx(data[0], ancilla[0])
    qc.cx(data[1], ancilla[0])
    qc.measure(ancilla[0], creg[0])
    
    qc.cx(data[1], ancilla[1])
    qc.cx(data[2], ancilla[1])
    qc.measure(ancilla[1], creg[1])

    qc.x(data[0]).c_if(creg, 0b10)
    qc.x(data[1]).c_if(creg, 0b11)
    qc.x(data[2]).c_if(creg, 0b01)


In [ ]:
from qiskit.circuit import ParameterVector
from qiskit.primitives import Estimator as AerEstimator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit.quantum_info import SparsePauliOp

# Define qubits
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

def create_protected_circuit(params):
    logical = QuantumRegister(1, 'logical')
    data = QuantumRegister(3, 'data')
    anc = QuantumRegister(2, 'ancilla')
    creg = ClassicalRegister(2, 'syndrome')
    qc = QuantumCircuit(logical, data, anc, creg)

    # Prepare logical state |0⟩
    encode(qc, logical, data)

    # Insert protected ansatz
    qc.compose(protected_ansatz(params, data), inplace=True)

    # Error correction
    measure_and_correct(qc, data, anc, creg)

    # Decode
    decode(qc, logical, data)

    return qc


In [ ]:
import numpy as np

# Hamiltonian
hamiltonian = SparsePauliOp.from_list([
    ("II", -1.052),
    ("IZ", 0.398),
    ("ZI", -0.398),
    ("ZZ", -0.011),
    ("XX", 0.181)
])

# Define param vector for ansatz
param_vec = ParameterVector('θ', 6)

# Custom ansatz with protection
def ansatz_wrapper(params):
    return create_protected_circuit(params)

# Noisy simulator
bit_flip_error = pauli_error([('X', 0.05), ('I', 0.95)])
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(bit_flip_error, ['cx'])
backend = AerSimulator(noise_model=noise_model)
estimator = AerEstimator(backend=backend)

# Optimizer
optimizer = SPSA(maxiter=100)

# VQE
vqe = VQE(ansatz=ansatz_wrapper, optimizer=optimizer, estimator=estimator, initial_point=np.random.rand(6))
result_protected = vqe.compute_minimum_eigenvalue(operator=hamiltonian)

print("Error-corrected energy:", result_protected.eigenvalue.real)


In [ ]:
import matplotlib.pyplot as plt

labels = ['Ideal', 'Noisy', 'Error-Corrected']
energies = [-1.857, -1.67, result_protected.eigenvalue.real]

plt.bar(labels, energies, color=['green', 'red', 'blue'])
plt.ylabel('Ground State Energy')
plt.title('Comparison of VQE Results')
plt.ylim([-1.9, -1.5])
plt.grid(axis='y')
plt.show()
